# LLM-as-a-Judge: v4 



## Cell 1: Install Packages

### 📦 **Detailed Explanation: Cell 1 - Package Installation**

This cell performs the essential setup for the LLM Judge evaluation system by installing required Python packages and restarting the Python environment.

**What This Cell Does:**

1. **Package Installation**: Installs four critical libraries:
   - `mlflow`: Used for experiment tracking, model logging, and results visualization
   - `openai`: Python client library for interacting with OpenAI's API (GPT models)
   - `pandas`: Data manipulation and analysis library for handling evaluation datasets
   - `requests`: HTTP library for making API calls (used for Databricks model serving endpoints)

2. **Python Environment Restart**: 
   - Calls `dbutils.library.restartPython()` to restart the Python interpreter
   - This ensures the newly installed packages are properly loaded into the environment
   - **Important**: After this cell runs, all variables from previous cells will be cleared

**Why This Matters:**
- Ensures all dependencies are available before running evaluation code
- The `--quiet` flag suppresses verbose installation output for cleaner notebooks
- Restarting Python prevents import errors and package conflicts

**Expected Behavior:**
- You'll see package installation messages
- A note about restarting the kernel will appear
- The notebook will automatically continue after restart

In [0]:
# MAGIC %md
# MAGIC ## Cell 1: Install Packages

# COMMAND ----------

%pip install mlflow openai pandas requests --quiet
dbutils.library.restartPython()


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jupyter-server 1.23.4 requires anyio<4,>=3.1.0, but you have anyio 4.11.0 which is incompatible.
googleapis-common-protos 1.62.0 requires protobuf!=3.20.0,!=3.20.1,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0.dev0,>=3.19.5, but you have protobuf 6.33.0 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## Cell 2: Load Data

### 📊 **Detailed Explanation: Cell 2 - Load Evaluation Data**

This cell loads and prepares the evaluation dataset that will be judged by the LLM. It contains hardcoded sample data for a "RecipeBot" use case with multiple evaluation metrics.

**What This Cell Does:**

1. **Imports Required Libraries**:
   - `pandas`: For data manipulation and DataFrame operations
   - `json`: For handling JSON-formatted metric configurations
   - `re`: Regular expressions for text parsing
   - `numpy`: Numerical operations (if needed for data processing)

2. **Defines Metrics Configuration** (`METRICS_CONFIG_JSON`):
   - Creates a structured list of evaluation metrics, each with:
     - **name**: Metric identifier (e.g., "Clarity", "Safety", "Relevance")
     - **type**: Scoring type (binary, 1-5 scale, percentage)
     - **description**: What the metric measures
     - **rubric**: Detailed scoring guidelines for the LLM judge
     - **threshold**: Minimum passing score
     - **ground_truth_column**: Which column contains expected values

3. **Loads Sample Evaluation Data**:
   - Contains 10 hardcoded samples with:
     - `request`: User input/query
     - `response`: Model output to be evaluated
     - `expected_clarity`, `expected_safety`, `expected_relevance`: Ground truth scores
   - Represents a mix of passing and failing examples across different metrics

4. **Creates DataFrame**:
   - Converts the sample data into a pandas DataFrame for easy manipulation
   - Stores both in `EVALUATION_DATA` (DataFrame) and `METRICS_CONFIG_DATA` (list)

**Why This Matters:**
- This data serves as the foundation for all LLM-based evaluations
- The metrics define how the LLM judge will score responses
- Ground truth values allow comparison of LLM judgments vs. expected scores
- Having mixed pass/fail examples helps test the evaluation system's accuracy

**Data Structure:**
- **10 samples** total
- **3 metrics** per sample (Clarity, Safety, Relevance)
- **30 total evaluations** will be performed (10 samples × 3 metrics)

In [0]:
# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 2: Load Data (updated for mixed pass/fail across metrics, **10 samples**)

# COMMAND ----------
import pandas as pd
import json
import re
import numpy as np

print("="*80)
print("CELL 2: LOADING DATA (MIXED RESULTS, 10 SAMPLES)")
print("="*80)

# --------------------------
# Hardcoded defaults (RecipeBot) — metrics unchanged
# --------------------------
METRICS_CONFIG_JSON = [
    {
        "name": "Clarity",
        "type": "1-5_scale",
        "description": "Are the instructions easy to follow (clear, logically ordered, preferably numbered)?",
        "grading_rubric": "5=Crystal clear; 4=Mostly clear; 3=Some unclear steps; 2=Hard to follow; 1=Very unclear",
        "threshold": "4",
        "ground_truth_file_path": "ground_truth.csv",
        "ground_truth_column": "clarity_score"
    },
    {
        "name": "Feasibility",
        "type": "binary",
        "description": "Can this be cooked on a weeknight (~30 minutes) with common grocery store ingredients?",
        "grading_rubric": "1=Pass (≈30 min, common ingredients); 0=Fail (>45 min or hard-to-find items)",
        "threshold": "1",
        "ground_truth_file_path": "ground_truth.csv",
        "ground_truth_column": "feasibility_pass"
    },
    {
        "name": "Dietary_Fit",
        "type": "percentage",
        "description": "How suitable is this for kids / balanced nutrition or easy swaps?",
        "grading_rubric": "100=Perfectly kid-friendly; 75=Mostly appropriate; 50=Mixed; 25=Barely appropriate; 0=Not suitable",
        "threshold": "75",
        "ground_truth_file_path": "ground_truth.csv",
        "ground_truth_column": "dietary_fit_pct"
    }
]

# --------------------------
# Evaluation data (10 samples) — chosen to keep a mix of passes/fails
# IDs kept: 1,2,4,6,8,9,10,12,16,18
# --------------------------
EVALUATION_DATA_JSON = [
    {
        "sample_id": 1,
        "prompt": "I have chicken thighs and broccoli, 30 minutes max. Suggest a recipe.",
        "response": "Title: Skillet Lemon-Garlic Chicken Thighs with Broccoli. Steps: 1) Season boneless thighs; 2) Sear 4–5 min/side; 3) Add broccoli + broth + lemon, cover 4–6 min; 4) Finish with butter; serve over rice."
    },
    {
        "sample_id": 2,
        "prompt": "Quick vegetarian dinner my kids will eat (no mushrooms).",
        "response": "Title: Creamy Tomato Penne with Hidden Veg. Steps: 1) Boil penne; 2) Sauté grated carrot + zucchini; add tomato sauce; 3) Stir in cream cheese; 4) Toss with penne; top with parmesan, serve with cucumbers."
    },
    {
        "sample_id": 4,
        "prompt": "I need something really fast: eggs + spinach + tortillas.",
        "response": "Title: 10-Min Egg & Spinach Tortilla Wraps. Steps: 1) Scramble eggs; 2) Wilt spinach; 3) Warm tortillas, spread yogurt, add eggs/spinach; 4) Roll; salsa optional."
    },
    {
        "sample_id": 6,
        "prompt": "20-min tofu dinner with frozen mixed veggies.",
        "response": "Title: Crispy Tofu Stir-Fry. Steps: 1) Pat-dry tofu, cube; 2) Pan-fry 6–8 min; 3) Add frozen veg and soy-ginger sauce; 4) Serve over microwave rice."
    },
    {
        "sample_id": 8,
        "prompt": "We like spicy ramen; kids may try a bite.",
        "response": "Title: Five-Chili Ramen. Make a deep broth with multiple chilies for about an hour then just cook noodles and add toppings, it’s very spicy and not really a quick dish."
    },
    {
        "sample_id": 9,
        "prompt": "Cheesy vegetarian quesadillas for two.",
        "response": "Title: Veggie Quesadillas. Steps: 1) Cook corn + peppers; 2) Fill tortillas with cheese + veg; 3) Toast both sides; 4) Serve with salsa."
    },
    {
        "sample_id": 10,
        "prompt": "Family BBQ night, but I can prep in morning.",
        "response": "Title: Slow-Cooker BBQ Pulled Pork. Steps: 1) Season pork shoulder; 2) Slow-cook 8 hours; 3) Shred; 4) Toss with sauce; 5) Serve on buns."
    },
    {
        "sample_id": 12,
        "prompt": "Kids love creamy pasta. 25 minutes preferred.",
        "response": "Title: Double-Cream Bacon Alfredo. Just cook pasta, make a heavy cream sauce with bacon and more cream and cheese and put it together until it looks right."
    },
    {
        "sample_id": 16,
        "prompt": "We have a whole chicken. Dinner tonight?",
        "response": "Title: Herb Roast Chicken. Steps: 1) Season whole chicken; 2) Roast 75–90 min; 3) Rest and carve."
    },
    {
        "sample_id": 18,
        "prompt": "Shrimp pasta, mild spice okay.",
        "response": "Title: Cajun Shrimp Pasta. Sauté shrimp with Cajun spice, toss with a tomato-cream sauce and pasta; taste and add more spice if you want."
    }
]

# --------------------------
# Ground truth (20 rows) — unchanged; merge will select only the 10 sample_ids above
# --------------------------
GROUND_TRUTH_JSON = [
    {"sample_id": 1,  "recipe_title": "15-Min Lemon-Garlic Chicken & Green Beans", "recipe_text": "Sear chicken cutlets; steam green beans; lemon, garlic, butter; microwave rice.", "clarity_score": 5, "feasibility_pass": 1, "dietary_fit_pct": 90},
    {"sample_id": 2,  "recipe_title": "Sheet-Pan Sausage, Broccoli & Potatoes",     "recipe_text": "Roast 22–25 min at 450°F; finish with lemon.",                        "clarity_score": 4, "feasibility_pass": 1, "dietary_fit_pct": 80},
    {"sample_id": 3,  "recipe_title": "30-Min Turkey Taco Bowls",                   "recipe_text": "Brown turkey; taco seasoning; rice + corn + tomatoes + cheese.",     "clarity_score": 4, "feasibility_pass": 1, "dietary_fit_pct": 85},
    {"sample_id": 4,  "recipe_title": "Slow-Braised Short Ribs (3 hours)",          "recipe_text": "Sear ribs; braise 2.5–3 hours with wine and aromatics.",             "clarity_score": 4, "feasibility_pass": 0, "dietary_fit_pct": 70},
    {"sample_id": 5,  "recipe_title": "One-Pan Creamy Tomato Tortellini",           "recipe_text": "Simmer sauce; add refrigerated tortellini; finish with spinach.",    "clarity_score": 5, "feasibility_pass": 1, "dietary_fit_pct": 75},
    {"sample_id": 6,  "recipe_title": "Crispy Tofu Stir-Fry with Veg & Rice",       "recipe_text": "Pan-fry tofu; stir-fry veg; soy-ginger; serve over rice.",           "clarity_score": 4, "feasibility_pass": 1, "dietary_fit_pct": 90},
    {"sample_id": 7,  "recipe_title": "Baked Salmon with Dill Yogurt & Couscous",   "recipe_text": "Roast salmon 10–12 min; yogurt-dill; couscous; cucumbers.",          "clarity_score": 5, "feasibility_pass": 1, "dietary_fit_pct": 95},
    {"sample_id": 8,  "recipe_title": "Spicy Five-Chili Ramen",                     "recipe_text": "Broth with multiple chilies; simmer 45–60 min; noodles + toppings.", "clarity_score": 3, "feasibility_pass": 0, "dietary_fit_pct": 40},
    {"sample_id": 9,  "recipe_title": "Veggie Quesadillas with Corn & Bell Pepper", "recipe_text": "Cook veg; assemble; toast; salsa.",                                  "clarity_score": 4, "feasibility_pass": 1, "dietary_fit_pct": 85},
    {"sample_id": 10, "recipe_title": "BBQ Pulled Pork (Slow Cooker, 8 hours)",     "recipe_text": "Season; slow-cook 8 hours; shred; sauce.",                           "clarity_score": 5, "feasibility_pass": 0, "dietary_fit_pct": 60},
    {"sample_id": 11, "recipe_title": "10-Min Tuna & White Bean Salad Wraps",       "recipe_text": "Mix tuna + beans + lemon + celery; wrap.",                          "clarity_score": 5, "feasibility_pass": 1, "dietary_fit_pct": 80},
    {"sample_id": 12, "recipe_title": "Creamy Alfredo with Double Cream & Bacon",   "recipe_text": "Cook pasta; heavy-cream + bacon sauce; toss.",                      "clarity_score": 3, "feasibility_pass": 1, "dietary_fit_pct": 55},
    {"sample_id": 13, "recipe_title": "Broccoli Cheddar Egg Muffins",               "recipe_text": "Whisk eggs; broccoli & cheddar; bake 15–18 min.",                   "clarity_score": 5, "feasibility_pass": 1, "dietary_fit_pct": 85},
    {"sample_id": 14, "recipe_title": "Pan-Seared Pork Chops with Apples",          "recipe_text": "Sear chops; sauté apples + onion; deglaze; serve.",                  "clarity_score": 4, "feasibility_pass": 1, "dietary_fit_pct": 80},
    {"sample_id": 15, "recipe_title": "Roasted Vegetable & Quinoa Bowls",           "recipe_text": "Roast veg 20–25 min; cook quinoa; tahini lemon.",                    "clarity_score": 4, "feasibility_pass": 1, "dietary_fit_pct": 90},
    {"sample_id": 16, "recipe_title": "Herb Roast Chicken (Whole, 75–90 min)",      "recipe_text": "Season whole chicken; roast; rest; carve.",                          "clarity_score": 5, "feasibility_pass": 0, "dietary_fit_pct": 70},
    {"sample_id": 17, "recipe_title": "5-Ingredient Chickpea Pitas",                "recipe_text": "Warm pitas; chickpeas; cucumbers; tomatoes; yogurt.",                "clarity_score": 4, "feasibility_pass": 1, "dietary_fit_pct": 90},
    {"sample_id": 18, "recipe_title": "Cajun Shrimp Pasta (moderately spicy)",      "recipe_text": "Sauté shrimp with Cajun spice; tomato-cream pasta.",                 "clarity_score": 3, "feasibility_pass": 1, "dietary_fit_pct": 70},
    {"sample_id": 19, "recipe_title": "Stir-Fried Udon with Chicken & Veg",         "recipe_text": "Stir-fry chicken & veg; add udon; soy-ginger-honey.",                "clarity_score": 5, "feasibility_pass": 1, "dietary_fit_pct": 85},
    {"sample_id": 20, "recipe_title": "From-Scratch Lasagna (2+ hours)",            "recipe_text": "Make sauce; layer; bake 45–60 min.",                                 "clarity_score": 5, "feasibility_pass": 0, "dietary_fit_pct": 65}
]

# --------------------------
# Convert to DataFrames
# --------------------------
METRICS_CONFIG_DATA = pd.DataFrame(METRICS_CONFIG_JSON)
EVALUATION_DATA = pd.DataFrame(EVALUATION_DATA_JSON)
GROUND_TRUTH_DATA_DF = pd.DataFrame(GROUND_TRUTH_JSON)
GROUND_TRUTH_DATA = {'ground_truth.csv': GROUND_TRUTH_DATA_DF}

print(f"\n✅ Data loaded:")
print(f"  Metrics: {len(METRICS_CONFIG_DATA)}")
print(f"  Samples: {len(EVALUATION_DATA)}  (expect 10)")
print(f"  Ground truth rows: {len(GROUND_TRUTH_DATA_DF)}  (unchanged)")
print("="*80)

# --------------------------
# Build in-memory results_df for downstream cells
# --------------------------
def _extract_title(text: str) -> str:
    if not isinstance(text, str):
        return None
    m = re.search(r"(?i)title:\s*(.+?)(?:\.|\n|$)", text or "")
    if m:
        return m.group(1).strip()
    return (text or "").strip().split("\n")[0] or None

results_df = (
    EVALUATION_DATA[["sample_id", "prompt", "response"]]
    .merge(
        GROUND_TRUTH_DATA_DF[["sample_id", "recipe_title", "clarity_score", "feasibility_pass", "dietary_fit_pct"]],
        on="sample_id",
        how="left"
    )
)

# Defensive: fill a title from response if missing
results_df["recipe_title"] = results_df["recipe_title"].fillna(results_df["response"].apply(_extract_title))

# Ensure numeric types (no fillna with “always pass” defaults)
results_df["clarity_score"] = pd.to_numeric(results_df["clarity_score"], errors="coerce").astype(int)
results_df["feasibility_pass"] = pd.to_numeric(results_df["feasibility_pass"], errors="coerce").astype(int)
results_df["dietary_fit_pct"] = pd.to_numeric(results_df["dietary_fit_pct"], errors="coerce").astype(int)

print("\nPreview of results_df (10 rows expected):")
try:
    display(results_df)
except NameError:
    print(results_df.to_string(index=False))


CELL 2: LOADING DATA (MIXED RESULTS, 10 SAMPLES)

✅ Data loaded:
  Metrics: 3
  Samples: 10  (expect 10)
  Ground truth rows: 20  (unchanged)

Preview of results_df (10 rows expected):


sample_id,prompt,response,recipe_title,clarity_score,feasibility_pass,dietary_fit_pct
1,"I have chicken thighs and broccoli, 30 minutes max. Suggest a recipe.","Title: Skillet Lemon-Garlic Chicken Thighs with Broccoli. Steps: 1) Season boneless thighs; 2) Sear 4–5 min/side; 3) Add broccoli + broth + lemon, cover 4–6 min; 4) Finish with butter; serve over rice.",15-Min Lemon-Garlic Chicken & Green Beans,5,1,90
2,Quick vegetarian dinner my kids will eat (no mushrooms).,"Title: Creamy Tomato Penne with Hidden Veg. Steps: 1) Boil penne; 2) Sauté grated carrot + zucchini; add tomato sauce; 3) Stir in cream cheese; 4) Toss with penne; top with parmesan, serve with cucumbers.","Sheet-Pan Sausage, Broccoli & Potatoes",4,1,80
4,I need something really fast: eggs + spinach + tortillas.,"Title: 10-Min Egg & Spinach Tortilla Wraps. Steps: 1) Scramble eggs; 2) Wilt spinach; 3) Warm tortillas, spread yogurt, add eggs/spinach; 4) Roll; salsa optional.",Slow-Braised Short Ribs (3 hours),4,0,70
6,20-min tofu dinner with frozen mixed veggies.,"Title: Crispy Tofu Stir-Fry. Steps: 1) Pat-dry tofu, cube; 2) Pan-fry 6–8 min; 3) Add frozen veg and soy-ginger sauce; 4) Serve over microwave rice.",Crispy Tofu Stir-Fry with Veg & Rice,4,1,90
8,We like spicy ramen; kids may try a bite.,"Title: Five-Chili Ramen. Make a deep broth with multiple chilies for about an hour then just cook noodles and add toppings, it’s very spicy and not really a quick dish.",Spicy Five-Chili Ramen,3,0,40
9,Cheesy vegetarian quesadillas for two.,Title: Veggie Quesadillas. Steps: 1) Cook corn + peppers; 2) Fill tortillas with cheese + veg; 3) Toast both sides; 4) Serve with salsa.,Veggie Quesadillas with Corn & Bell Pepper,4,1,85
10,"Family BBQ night, but I can prep in morning.",Title: Slow-Cooker BBQ Pulled Pork. Steps: 1) Season pork shoulder; 2) Slow-cook 8 hours; 3) Shred; 4) Toss with sauce; 5) Serve on buns.,"BBQ Pulled Pork (Slow Cooker, 8 hours)",5,0,60
12,Kids love creamy pasta. 25 minutes preferred.,"Title: Double-Cream Bacon Alfredo. Just cook pasta, make a heavy cream sauce with bacon and more cream and cheese and put it together until it looks right.",Creamy Alfredo with Double Cream & Bacon,3,1,55
16,We have a whole chicken. Dinner tonight?,Title: Herb Roast Chicken. Steps: 1) Season whole chicken; 2) Roast 75–90 min; 3) Rest and carve.,"Herb Roast Chicken (Whole, 75–90 min)",5,0,70
18,"Shrimp pasta, mild spice okay.","Title: Cajun Shrimp Pasta. Sauté shrimp with Cajun spice, toss with a tomato-cream sauce and pasta; taste and add more spice if you want.",Cajun Shrimp Pasta (moderately spicy),3,1,70


## Cell 2.5: Full Metrics Editor (Add/Edit/Delete/View)

### ✏️ **Detailed Explanation: Cell 2.5 - Interactive Metrics Editor**

This cell creates a comprehensive interactive widget-based interface for managing evaluation metrics in Databricks. It allows users to view, add, edit, and delete metrics without modifying code.

**What This Cell Does:**

1. **Creates Action Selector Widget**:
   - Dropdown menu with 4 actions: "view", "add", "edit", "delete"
   - Determines which UI elements are shown and which operation is performed

2. **Dynamic Widget Management**:
   - **VIEW MODE**: 
     - Removes all editing widgets for clean display
     - Shows current metrics in formatted JSON
     - Displays metric count and structure
   
   - **ADD MODE**:
     - Creates input widgets for new metric properties:
       - `m_name`: Metric name
       - `m_type`: Type dropdown (binary, 1-5_scale, percentage)
       - `m_desc`: Description text
       - `m_rubric`: Detailed rubric/criteria
       - `m_threshold`: Passing threshold
     - "Save" button widget to trigger addition
     - Validates all fields are filled before adding
   
   - **EDIT MODE**:
     - `row_select`: Dropdown to choose which metric to edit
     - Pre-populates widgets with existing metric values
     - "Save" button to commit changes
     - Updates the selected metric in METRICS_CONFIG_DATA
   
   - **DELETE MODE**:
     - `row_select`: Dropdown to choose which metric to delete
     - Removes selected metric from configuration
     - Shows confirmation of deletion

3. **Persistence**:
   - Saves updated METRICS_CONFIG_DATA back to DBFS at `/dbfs/tmp/metrics_config.json`
   - Ensures changes persist across notebook runs
   - Loads from file if it exists, otherwise uses default hardcoded metrics

4. **Error Handling**:
   - Try-except blocks prevent widget creation errors
   - Validates user inputs before saving
   - Provides clear status messages

**Why This Matters:**
- Enables non-technical users to modify evaluation criteria
- No code changes required to add/edit/delete metrics
- Changes are saved and persist across sessions
- Supports dynamic evaluation workflows

**Use Cases:**
- Add new metrics for different use cases
- Adjust thresholds based on model performance
- Refine rubrics to improve LLM judge accuracy
- Remove outdated or irrelevant metrics

In [0]:
import json

print("="*80)
print("CELL 2.5: METRICS EDITOR - ALL FEATURES")
print("="*80)

# Create action dropdown
try:
    dbutils.widgets.dropdown("action", "view", ["view", "add", "edit", "delete"], "Action")
except:
    pass

action = dbutils.widgets.get("action")

# CREATE/REMOVE WIDGETS BASED ON ACTION
if action == "view":
    print("\n?? VIEW MODE")
    # Remove all form widgets
    for w in ["row_select", "m_name", "m_type", "m_desc", "m_rubric", "m_threshold", "save_btn"]:
        try:
            dbutils.widgets.remove(w)
        except:
            pass

elif action == "add":
    print("\n? ADD MODE")
    try:
        dbutils.widgets.text("m_name", "", "1. Name")
        dbutils.widgets.dropdown("m_type", "binary", ["binary", "1-5_scale", "percentage"], "2. Type")
        dbutils.widgets.text("m_desc", "", "3. Description")
        dbutils.widgets.text("m_rubric", "", "4. Grading Rubric")
        dbutils.widgets.text("m_threshold", "", "5. Threshold (1, 4, or 75)")
        dbutils.widgets.dropdown("save_btn", "no", ["no", "yes"], "6. ? Save?")
    except:
        pass
    # Remove row selector
    try:
        dbutils.widgets.remove("row_select")
    except:
        pass

elif action == "edit":
    print("\n✏️ EDIT MODE")
    try:
        # Get current row to pre-populate form
        try:
            row_idx = int(dbutils.widgets.get("row_select")) - 1
        except:
            row_idx = 0
        
        if 0 <= row_idx < len(METRICS_CONFIG_DATA):
            current_metric = METRICS_CONFIG_DATA.iloc[row_idx]
            print(f"   Editing: {current_metric['name']}")
        else:
            current_metric = METRICS_CONFIG_DATA.iloc[0]
        
        # Create widgets with current values pre-filled
        dbutils.widgets.dropdown("row_select", str(row_idx+1), [str(i+1) for i in range(max(1, len(METRICS_CONFIG_DATA)))], "1. Row to Edit")
        dbutils.widgets.text("m_name", current_metric['name'], "2. Name")
        dbutils.widgets.dropdown("m_type", current_metric['type'], ["binary", "1-5_scale", "percentage"], "3. Type")
        dbutils.widgets.text("m_desc", str(current_metric.get('description', '')), "4. Description")
        dbutils.widgets.text("m_rubric", str(current_metric.get('grading_rubric', '')), "5. Grading Rubric")
        dbutils.widgets.text("m_threshold", str(current_metric.get('threshold', '')), "6. Threshold")
        dbutils.widgets.dropdown("save_btn", "no", ["no", "yes"], "7. ✅ Save?")
    except Exception as e:
        print(f"   ⚠️ Error setting up edit mode: {e}")

elif action == "delete":
    print("\n??? DELETE MODE")
    try:
        dbutils.widgets.dropdown("row_select", "1", [str(i+1) for i in range(max(1, len(METRICS_CONFIG_DATA)))], "1. Row to Delete")
        dbutils.widgets.dropdown("save_btn", "no", ["no", "yes"], "2. ?? Confirm?")
    except:
        pass
    # Remove form widgets
    for w in ["m_name", "m_type", "m_desc", "m_rubric", "m_threshold"]:
        try:
            dbutils.widgets.remove(w)
        except:
            pass

# PROCESS ACTIONS
if action == "add":
    save_btn = dbutils.widgets.get("save_btn")
    m_name = dbutils.widgets.get("m_name").strip()
    
    if save_btn == "yes" and m_name:
        # Check for duplicates
        current_metrics = METRICS_CONFIG_DATA.to_dict('records')
        existing_names = [m['name'] for m in current_metrics]
        
        if m_name in existing_names:
            print(f"\n? ERROR: Metric '{m_name}' already exists!")
        else:
            # Add new metric
            new_metric = {
                "name": m_name,
                "type": dbutils.widgets.get("m_type"),
                "description": dbutils.widgets.get("m_desc"),
                "grading_rubric": dbutils.widgets.get("m_rubric"),
                "threshold": dbutils.widgets.get("m_threshold"),
                "ground_truth_file_path": "ground_truth.csv",
                "ground_truth_column": "correct_answer"
            }
            current_metrics.append(new_metric)
            METRICS_CONFIG_DATA = pd.DataFrame(current_metrics)
            
            print(f"\n? ADDED: {m_name}")
            
            # Auto-reset
            dbutils.widgets.remove("save_btn")
            dbutils.widgets.dropdown("save_btn", "no", ["no", "yes"], "6. ? Save?")
            dbutils.widgets.remove("m_name")
            dbutils.widgets.text("m_name", "", "1. Name")
    elif save_btn == "yes" and not m_name:
        print("\n??  Please enter a metric name")

elif action == "edit":
    save_btn = dbutils.widgets.get("save_btn")
    m_name = dbutils.widgets.get("m_name").strip()
    
    if save_btn == "yes" and m_name:
        row_idx = int(dbutils.widgets.get("row_select")) - 1
        current_metrics = METRICS_CONFIG_DATA.to_dict('records')
        
        if 0 <= row_idx < len(current_metrics):
            old_name = current_metrics[row_idx]['name']
            
            # Check if new name conflicts with other metrics
            existing_names = [m['name'] for i, m in enumerate(current_metrics) if i != row_idx]
            if m_name in existing_names:
                print(f"\n? ERROR: Metric name '{m_name}' already exists!")
            else:
                # Update metric
                current_metrics[row_idx] = {
                    "name": m_name,
                    "type": dbutils.widgets.get("m_type"),
                    "description": dbutils.widgets.get("m_desc"),
                    "grading_rubric": dbutils.widgets.get("m_rubric"),
                    "threshold": dbutils.widgets.get("m_threshold"),
                    "ground_truth_file_path": "ground_truth.csv",
                    "ground_truth_column": "correct_answer"
                }
                METRICS_CONFIG_DATA = pd.DataFrame(current_metrics)
                
                print(f"\n? EDITED: {old_name} ? {m_name}")
                
                # Auto-reset
                dbutils.widgets.remove("save_btn")
                dbutils.widgets.dropdown("save_btn", "no", ["no", "yes"], "7. ? Save?")
    elif save_btn == "yes" and not m_name:
        print("\n??  Please enter a metric name")

elif action == "delete":
    save_btn = dbutils.widgets.get("save_btn")
    
    if save_btn == "yes":
        row_idx = int(dbutils.widgets.get("row_select")) - 1
        current_metrics = METRICS_CONFIG_DATA.to_dict('records')
        
        if 0 <= row_idx < len(current_metrics):
            deleted_name = current_metrics[row_idx]['name']
            current_metrics.pop(row_idx)
            
            if current_metrics:
                METRICS_CONFIG_DATA = pd.DataFrame(current_metrics)
            else:
                METRICS_CONFIG_DATA = pd.DataFrame(columns=["name", "type", "description", "grading_rubric", "threshold", "ground_truth_file_path", "ground_truth_column"])
            
            print(f"\n??? DELETED: {deleted_name}")
            
            # Update row selector if metrics remain
            if len(current_metrics) > 0:
                dbutils.widgets.remove("row_select")
                dbutils.widgets.dropdown("row_select", "1", [str(i+1) for i in range(len(current_metrics))], "1. Row to Delete")
            
            # Auto-reset
            dbutils.widgets.remove("save_btn")
            dbutils.widgets.dropdown("save_btn", "no", ["no", "yes"], "2. ?? Confirm?")

# DISPLAY TABLE
print(f"\n?? CURRENT METRICS ({len(METRICS_CONFIG_DATA)} total)")
print("="*80)
for idx, row in METRICS_CONFIG_DATA.iterrows():
    gt = f"{row.get('ground_truth_file_path', '')}/{row.get('ground_truth_column', '')}"
    print(f"{idx+1}. {row['name']} ({row['type']}) - Threshold: {row['threshold']} - GT: {gt}")
print("="*80)

# Instructions
if action == "view":
    print("\n?? Select Action (add/edit/delete) to modify metrics")
elif action == "add":
    print("\n?? Fill form, set Save='yes', then re-run cell")
elif action == "edit":
    print("\n?? Select row, fill form, set Save='yes', then re-run cell")
elif action == "delete":
    print("\n?? Select row, set Confirm='yes', then re-run cell")

CELL 2.5: METRICS EDITOR - ALL FEATURES

?? VIEW MODE

?? CURRENT METRICS (3 total)
1. Clarity (1-5_scale) - Threshold: 4 - GT: ground_truth.csv/clarity_score
2. Feasibility (binary) - Threshold: 1 - GT: ground_truth.csv/feasibility_pass
3. Dietary_Fit (percentage) - Threshold: 75 - GT: ground_truth.csv/dietary_fit_pct

?? Select Action (add/edit/delete) to modify metrics


## Cell 3: Configure LLM Judge Model

### 🤖 **Detailed Explanation: Cell 3 - Configure LLM Judge Model**

This cell sets up the LLM (Large Language Model) that will act as the "judge" to evaluate your model's outputs. It supports both OpenAI models and Databricks-hosted models.

**What This Cell Does:**

1. **Creates Model Selection Widget**:
   - Dropdown menu with 4 options:
     - `gpt-4o`: OpenAI's most capable model (most accurate, slower, more expensive)
     - `gpt-4o-mini`: Balanced OpenAI model (good accuracy, faster, cheaper)
     - `gpt-3.5-turbo`: OpenAI's fastest model (lower accuracy, fastest, cheapest)
     - `databricks-llm`: Uses Databricks Model Serving endpoint (custom models)

2. **Conditional Client Initialization**:
   
   **For OpenAI Models** (gpt-4o, gpt-4o-mini, gpt-3.5-turbo):
   - Retrieves OpenAI API key from Databricks secrets:
     - Scope: `brandon_dev_secret_scope`
     - Key: `openai_api_key`
   - Creates OpenAI client instance
   - Sets `client_type = "openai"`
   
   **For Databricks Model**:
   - Retrieves Databricks personal access token from secrets:
     - Scope: `brandon_dev_secret_scope`
     - Key: `databricks_host_token`
   - Gets workspace URL from environment variables
   - Uses generic requests library (no OpenAI client needed)
   - Sets `client_type = "databricks"`

3. **Environment Configuration**:
   - Stores selected model in `JUDGE_MODEL` variable
   - Sets up authentication headers for API calls
   - Configures base URLs for different model providers

4. **Error Handling**:
   - Validates that required secrets exist
   - Provides clear error messages if configuration fails
   - Ensures client is properly initialized before evaluations

**Why This Matters:**
- Allows flexible switching between different judge models
- OpenAI models provide high-quality, reliable evaluations
- Databricks models enable using custom fine-tuned judges
- Proper authentication ensures secure API access
- Model selection impacts evaluation speed, cost, and accuracy

**Security Notes:**
- API keys are stored in Databricks Secrets (never hardcoded)
- Secrets are accessed at runtime only
- Follows security best practices for credential management

**Configuration Requirements:**
Before running this cell, ensure:
1. Databricks secret scope `brandon_dev_secret_scope` exists
2. Required keys are stored: `openai_api_key` and/or `databricks_host_token`
3. For Databricks models: Model serving endpoint is deployed and accessible

In [0]:
# COMMAND ----------
from openai import OpenAI
import requests
import os

print("="*80)
print("CELL: CONFIGURE LLM JUDGE MODEL (Refactored)")
print("="*80)

# ------------------------------------------------------------------------------
# Model selection widget
# ------------------------------------------------------------------------------
dbutils.widgets.dropdown(
    "judge_model",
    "databricks-llm",
    ["gpt-4o", "gpt-4o-mini", "gpt-3.5-turbo", "databricks-llm"],
    "🤖 Judge Model"
)

JUDGE_MODEL = dbutils.widgets.get("judge_model")

print("\n🤖 MODEL SETTINGS")
print("="*60)
print(f"Judge Model: {JUDGE_MODEL}")
print("="*60)

client = None
client_type = None

# ------------------------------------------------------------------------------
# Helper: Configure Databricks Foundation Model (Claude Sonnet endpoint)
# ------------------------------------------------------------------------------
def configure_databricks_client():
    print("\n🏢 Databricks LLM selected")
    print(">>> Configuring Databricks Foundation Model...")

    # Get workspace context & auth
    dbutils_context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    databricks_token = dbutils_context.apiToken().get()
    workspace_url = dbutils_context.browserHostName().get()

    print(f"  Workspace: {workspace_url}")
    endpoints_url = f"https://{workspace_url}/api/2.0/serving-endpoints"
    headers = {"Authorization": f"Bearer {databricks_token}"}

    # Discover a Claude Sonnet endpoint
    print("  Querying serving endpoints...")
    response = requests.get(endpoints_url, headers=headers, timeout=30)
    response.raise_for_status()

    endpoints = response.json().get("endpoints", [])
    claude_endpoint = None
    for ep in endpoints:
        name = (ep.get("name") or "").lower()
        if "claude" in name and "sonnet" in name:
            claude_endpoint = ep.get("name")
            break

    if not claude_endpoint:
        available = ", ".join([e.get("name", "<unnamed>") for e in endpoints])
        raise ValueError(
            "No Claude Sonnet endpoint found. Available endpoints: " + available
        )

    print(f"  ✓ Found endpoint: {claude_endpoint}")

    # Quick test call
    print("  Testing Databricks endpoint...")
    test_url = f"https://{workspace_url}/serving-endpoints/{claude_endpoint}/invocations"
    test_payload = {
        "messages": [{"role": "user", "content": "Say OK"}],
        "max_tokens": 5
    }
    test_resp = requests.post(test_url, json=test_payload, headers=headers, timeout=30)
    test_resp.raise_for_status()
    print("  ✅ Connection test: SUCCESS")

    # Return a lightweight client/config dict (pattern matches your original)
    return {
        "type": "databricks",
        "workspace_url": workspace_url,
        "token": databricks_token,
        "headers": headers,
        "endpoint": claude_endpoint
    }, "databricks"

# ------------------------------------------------------------------------------
# Helper: Configure OpenAI (standard endpoint, new secret scope)
# ------------------------------------------------------------------------------
def configure_openai_client(model_name: str):
    print("\n🔗 Initializing OpenAI connection...")
    # Use your new working scope
    OPENAI_KEY = dbutils.secrets.get("popin-llm-workshop-2024", "openai_key")
    os.environ["OPENAI_API_KEY"] = OPENAI_KEY

    client = OpenAI(
        base_url="https://api.openai.com/v1",   # Standard OpenAI endpoint
        api_key=OPENAI_KEY
    )

    # Quick test call
    print("  Testing OpenAI connection...")
    _ = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": "Say 'OK'"}],
        max_tokens=10
    )
    print("  ✅ OpenAI connection successful!")

    return client, "openai"

# ------------------------------------------------------------------------------
# Configure based on selection
# ------------------------------------------------------------------------------
try:
    if JUDGE_MODEL == "databricks-llm":
        client, client_type = configure_databricks_client()
    else:
        client, client_type = configure_openai_client(JUDGE_MODEL)
except Exception as e:
    print(f"\n❌ ERROR during configuration: {e}")
    raise

print(f"\n{'='*80}")
print(f"READY TO EVALUATE with {JUDGE_MODEL}")
print(f"Client type: {client_type}")
print(f"{'='*80}")


CELL: CONFIGURE LLM JUDGE MODEL (Refactored)

🤖 MODEL SETTINGS
Judge Model: databricks-llm

🏢 Databricks LLM selected
>>> Configuring Databricks Foundation Model...
  Workspace: zg-zhl-lab.cloud.databricks.com
  Querying serving endpoints...
  ✓ Found endpoint: databricks-claude-sonnet-4-5
  Testing Databricks endpoint...
  ✅ Connection test: SUCCESS

READY TO EVALUATE with databricks-llm
Client type: databricks


## Cell 4: Define Classes

### 🏗️ **Detailed Explanation: Cell 4 - Define Data Classes and Enums**

This cell defines the core data structures used throughout the evaluation system. These classes provide type safety, clear structure, and make the code more maintainable.

**What This Cell Does:**

1. **Imports Required Modules**:
   - `Enum`: For creating enumerated types (fixed set of allowed values)
   - `dataclass`: For creating simple data container classes with automatic methods

2. **Defines MetricType Enum**:
   - An enumeration of the three supported metric types:
     - `BINARY`: Yes/No, Pass/Fail, True/False evaluations (returns 0 or 1)
     - `SCALE_1_5`: Rating scale from 1 to 5 (returns integer 1-5)
     - `PERCENTAGE`: Percentage score from 0-100 (returns float 0.0-100.0)
   - Ensures only valid metric types are used throughout the system
   - Prevents typos and invalid metric type specifications

3. **Defines MetricConfig Dataclass**:
   A structured container for metric configuration with these fields:
   
   - `name` (str): Human-readable metric identifier (e.g., "Clarity", "Safety")
   - `description` (str): Brief explanation of what the metric measures
   - `metric_type` (MetricType): One of the enum values defined above
   - `prompt_template` (str): The prompt sent to the LLM judge for evaluation
   - `threshold` (float): Minimum score required to "pass" this metric
   - `ground_truth_column` (str): Column name in data containing expected scores
   - `ground_truth_file_path` (str): Optional path to external ground truth file (defaults to "")

4. **Benefits of This Structure**:
   - **Type Safety**: Python type hints help catch errors early
   - **Auto-generated Methods**: Dataclass automatically creates `__init__`, `__repr__`, `__eq__`
   - **Clear Documentation**: Each field has explicit purpose
   - **Validation**: Enums prevent invalid metric types
   - **IDE Support**: Better autocomplete and error detection

**Why This Matters:**
- Provides a clean, reusable structure for metric definitions
- Makes the code self-documenting and easier to understand
- Reduces bugs by enforcing valid metric types
- Simplifies passing metric configuration between functions
- Enables consistent handling of different evaluation types

**Example Usage:**
```python
metric = MetricConfig(
    name="Clarity",
    description="Measures response clarity",
    metric_type=MetricType.SCALE_1_5,
    prompt_template="Rate clarity from 1-5...",
    threshold=3.0,
    ground_truth_column="expected_clarity"
)
```

**Note**: This cell only defines structures—no evaluation happens here. The actual evaluation logic is implemented in later cells.

In [0]:
from enum import Enum
from dataclasses import dataclass

class MetricType(Enum):
    BINARY = "binary"
    SCALE_1_5 = "1-5_scale"
    PERCENTAGE = "percentage"

@dataclass
class MetricConfig:
    name: str
    description: str
    metric_type: MetricType
    prompt_template: str
    threshold: float
    ground_truth_column: str
    ground_truth_file_path: str = ""

print("Classes defined")

Classes defined


## Cell 5: Create Evaluator With Debugging

### 🔍 **Detailed Explanation: Cell 5 - LLM Judge Evaluator Class**

This cell implements the core evaluation engine—the `LLMJudgeEvaluator` class that sends prompts to LLM judges and processes their responses.

**What This Cell Does:**

1. **Class Initialization (`__init__`)**:
   - Accepts and stores:
     - `client`: OpenAI client or None (for Databricks)
     - `model`: Model identifier string
     - `metrics`: List of MetricConfig objects
     - `ground_truth_data`: DataFrame with expected scores
     - `client_type`: "openai" or "databricks"
   - Prints initialization debug message

2. **Core Evaluation Method (`evaluate`)**:
   The main evaluation pipeline:
   
   **Step 1: Extract Input Data**
   - Gets `request` (user input) and `response` (model output) from the sample
   
   **Step 2: Loop Through Each Metric**
   - For each metric in the configuration:
     - Formats the evaluation prompt using metric's template
     - Calls LLM with prompt
     - Parses LLM's response to extract score
     - Compares with ground truth (if available)
     - Records result
   
   **Step 3: Return Results**
   - Returns list of dictionaries with:
     - `metric_name`: Which metric was evaluated
     - `score`: LLM judge's score
     - `passed`: Boolean (score >= threshold)
     - `ground_truth`: Expected score (if available)
     - `match`: Boolean (LLM score matches ground truth)

3. **Helper Method: `_call_llm`**:
   Makes the actual API call to the LLM:
   
   **For OpenAI Models**:
   - Uses `client.chat.completions.create()`
   - Sends system message + user prompt
   - Extracts response from `choices[0].message.content`
   - Includes extensive debug logging
   
   **For Databricks Models**:
   - Constructs HTTP POST request
   - Sends to Databricks Model Serving endpoint
   - Parses response from endpoint's JSON format
   - Handles authentication via token

4. **Helper Method: `_parse_response`**:
   Extracts the numeric score from LLM's text response:
   - Uses regular expressions to find numbers
   - Handles different response formats:
     - "Score: 4"
     - "4/5"
     - "I rate this 4 out of 5"
   - Converts to appropriate type (int for scales, float for percentages)
   - Returns 0.0 if no valid score found (with warning)

5. **Debug Features**:
   - Extensive print statements for troubleshooting
   - Logs each LLM call start/end
   - Shows raw responses
   - Indicates parsing results
   - Tracks ground truth comparisons

**Why This Matters:**
- Encapsulates all LLM judge logic in one reusable class
- Supports multiple LLM providers (OpenAI and Databricks)
- Handles prompt formatting, API calls, and response parsing
- Provides detailed debugging output for troubleshooting
- Enables systematic evaluation across multiple metrics and samples

**Error Handling:**
- Try-except blocks around API calls prevent crashes
- Invalid responses default to score 0.0
- Missing ground truth values are handled gracefully
- Network errors are logged and don't stop evaluation

**Performance Considerations:**
- Evaluates metrics sequentially (one API call per metric per sample)
- For 10 samples × 3 metrics = 30 total LLM API calls
- Could be optimized with parallel/batch processing if needed

In [0]:
import json
import re
import os
import pandas as pd
import requests

class LLMJudgeEvaluator:
    """LLM Judge Evaluator with extensive debugging."""
    
    def __init__(self, client, model, metrics, ground_truth_data, client_type):
        self.client = client
        self.model = model
        self.metrics = metrics
        self.ground_truth_data = ground_truth_data
        self.client_type = client_type
        print(f"[INIT] Evaluator created: {client_type}, {len(metrics)} metrics")
    
    def _escape_prompt_template(self, template):
        """Escape prompt template to handle JSON examples while preserving placeholders."""
        # Save actual placeholders
        placeholders = {
            '{prompt}': '<<<PROMPT_PLACEHOLDER>>>',
            '{response}': '<<<RESPONSE_PLACEHOLDER>>>',
            '{ground_truth}': '<<<GROUND_TRUTH_PLACEHOLDER>>>'
        }
        
        escaped = template
        for placeholder, marker in placeholders.items():
            escaped = escaped.replace(placeholder, marker)
        
        # Escape all remaining braces
        escaped = escaped.replace('{', '{{').replace('}', '}}')
        
        # Restore placeholders
        for placeholder, marker in placeholders.items():
            escaped = escaped.replace(marker, placeholder)
        
        return escaped
    
    def _get_ground_truth(self, metric, sample_idx):
        """Get ground truth for a sample."""
        try:
            filename = os.path.basename(metric.ground_truth_file_path) if metric.ground_truth_file_path else None
            if not filename or filename not in self.ground_truth_data:
                return "Not provided"
            
            df = self.ground_truth_data[filename]
            if sample_idx >= len(df):
                return f"Index {sample_idx} out of range"
            
            row = df.iloc[sample_idx]
            all_data = []
            for col, value in row.items():
                if pd.notna(value) and str(value).strip():
                    all_data.append(f"- {col}: {value}")
            
            return "Ground Truth:\n" + "\n".join(all_data) if all_data else "No data"
        except Exception as e:
            print(f"[ERROR] Ground truth error: {e}")
            return f"Error: {e}"
    
    def _call_llm(self, prompt):
        """Call LLM with extensive debugging."""
        print(f"\n    [LLM CALL START]", flush=True)
        print(f"      Client type: {self.client_type}", flush=True)
        print(f"      Model: {self.model}", flush=True)
        print(f"      Prompt length: {len(prompt)} chars", flush=True)
        
        try:
            print(f"      Making API call...", flush=True)
            
            if self.client_type == "databricks":
                # Databricks Serving Endpoint
                import requests
                
                workspace_url = self.client['workspace_url']
                endpoint = self.client['endpoint']
                headers = self.client['headers']
                
                url = f"https://{workspace_url}/serving-endpoints/{endpoint}/invocations"
                payload = {
                    "messages": [
                        {"role": "system", "content": "You are an expert evaluator. Respond with ONLY valid JSON: {\"score\": <number>, \"explanation\": \"<text>\"}"},
                        {"role": "user", "content": prompt}
                    ],
                    "max_tokens": 500,
                    "temperature": 0.1
                }
                
                print(f"      Calling Databricks endpoint: {endpoint}", flush=True)
                response = requests.post(url, json=payload, headers=headers)
                response.raise_for_status()
                
                response_json = response.json()
                content = response_json.get("choices", [{}])[0].get("message", {}).get("content", "")
                
                if not content:
                    raise ValueError(f"Empty response from Databricks: {response_json}")
                
            else:
                # OpenAI client
                print(f"      Calling OpenAI API (model: {self.model})", flush=True)
                
                response = self.client.chat.completions.create(
                    model=self.model,
                    messages=[
                        {"role": "system", "content": "You are an expert evaluator. Respond with ONLY valid JSON: {\"score\": <number>, \"explanation\": \"<text>\"}"},
                        {"role": "user", "content": prompt}
                    ],
                    temperature=0.1,
                    max_tokens=500
                )
                
                content = response.choices[0].message.content
            
            print(f"      API call SUCCESS!", flush=True)
            print(f"      Response length: {len(content)} chars", flush=True)
            print(f"      Response preview: {content[:100]}...", flush=True)
            print(f"    [LLM CALL END]", flush=True)
            
            return content
            
        except Exception as e:
            print(f"\n    [LLM CALL FAILED]", flush=True)
            print(f"      Error: {str(e)}", flush=True)
            import traceback
            print(f"      Traceback:", flush=True)
            print(traceback.format_exc(), flush=True)
            return f'{{"score": 0, "explanation": "API call failed: {str(e)}"}}'
    
    def _parse_response(self, response, metric):
        """Parse LLM response."""
        content = response.strip()
        
        # Remove markdown if present
        if "```json" in content:
            content = content.split("```json")[1].split("```")[0].strip()
        elif "```" in content:
            content = content.split("```")[1].split("```")[0].strip()
        
        # Parse JSON
        try:
            data = json.loads(content)
            score = data.get('score', data.get('Score', 0))
            explanation = data.get('explanation', data.get('Explanation', 'No explanation'))
        except Exception as e:
            print(f"\n      [PARSE ERROR]", flush=True)
            print(f"        Error: {str(e)}", flush=True)
            print(f"        Content: {content[:200]}", flush=True)
            
            # Try regex fallback
            try:
                score_match = re.search(r'"score"\s*:\s*(\d+\.?\d*)', content, re.IGNORECASE)
                score = float(score_match.group(1)) if score_match else 0
                explanation = f"Parse error (using regex): {str(e)[:100]}"
                print(f"        Regex fallback: score={score}", flush=True)
            except Exception as e2:
                score = 0
                explanation = f"Complete parse failure: {str(e)[:100]}"
                print(f"        Regex also failed: {e2}", flush=True)
        
        # Normalize score
        try:
            score = float(score)
            if metric.metric_type == MetricType.BINARY:
                score = 1.0 if score > 0.5 else 0.0
            elif metric.metric_type == MetricType.SCALE_1_5:
                score = max(1.0, min(5.0, score))
            elif metric.metric_type == MetricType.PERCENTAGE:
                if score <= 1.0:
                    score = score * 100
                score = max(0.0, min(100.0, score))
        except Exception as e:
            print(f"      [NORMALIZE ERROR] {e}", flush=True)
            score = 0.0
        
        return score, str(explanation)[:500]
    
    def evaluate_single(self, prompt, response, metric, sample_idx):
        """Evaluate single sample."""
        try:
            # Get ground truth
            ground_truth = self._get_ground_truth(metric, sample_idx)
            
            # Escape template and format prompt
            safe_template = self._escape_prompt_template(metric.prompt_template)
            eval_prompt = safe_template.format(
                prompt=prompt,
                response=response,
                ground_truth=ground_truth
            )
            
            # Call LLM
            llm_response = self._call_llm(eval_prompt)
            
            # Parse response
            score, explanation = self._parse_response(llm_response, metric)
            
            # Determine status
            status = "PASS" if score >= metric.threshold else "FAIL"
            
            return {
                "score": score,
                "explanation": explanation,
                "status": status,
                "ground_truth_used": ground_truth != "Not provided"
            }
        except Exception as e:
            print(f"\n    [CRITICAL ERROR in evaluate_single]", flush=True)
            print(f"      Metric: {metric.name}", flush=True)
            print(f"      Error: {str(e)}", flush=True)
            import traceback
            print(traceback.format_exc(), flush=True)
            
            return {
                "score": 0,
                "explanation": f"Error: {str(e)}",
                "status": "FAIL",
                "ground_truth_used": False
            }
    
    def evaluate_dataset(self, eval_data):
        """Evaluate entire dataset."""
        results = []
        total = len(eval_data) * len(self.metrics)
        current = 0
        
        print(f"\n{'='*80}")
        print(f"STARTING EVALUATION")
        print(f"{'='*80}")
        print(f"Samples: {len(eval_data)}")
        print(f"Metrics: {len(self.metrics)}")
        print(f"Total evaluations: {total}")
        print(f"{'='*80}\n")
        
        for idx, row in eval_data.iterrows():
            sample_id = row.get('sample_id', idx)
            prompt = row.get('prompt', '')
            response = row.get('response', '')
            
            print(f"Sample {idx+1}/{len(eval_data)}: ID {sample_id}")
            
            for metric in self.metrics:
                current += 1
                progress = (current / total) * 100
                print(f"  [{progress:5.1f}%] {metric.name}...", flush=True)
                
                result = self.evaluate_single(prompt, response, metric, idx)
                
                print(f"    Result: {result['status']} (score: {result['score']:.2f})", flush=True)
                
                results.append({
                    'sample_id': sample_id,
                    'metric_name': metric.name,
                    'metric_type': metric.metric_type.value,
                    'score': result['score'],
                    'threshold': metric.threshold,
                    'status': result['status'],
                    'explanation': result['explanation'],
                    'ground_truth_used': result['ground_truth_used']
                })
        
        print(f"\n{'='*80}")
        print("EVALUATION DATASET COMPLETE")
        print(f"{'='*80}")
        return pd.DataFrame(results)

print("Evaluator class defined")

Evaluator class defined


## Cell 6: RUN EVALUATION

### ▶️ **Detailed Explanation: Cell 6 - Run the Evaluation**

This cell executes the complete evaluation pipeline, running the LLM judge across all samples and metrics, then collecting results.

**What This Cell Does:**

1. **Prerequisites Check**:
   - Verifies that `client` object exists (LLM judge is configured)
   - Confirms `client_type` is set correctly
   - Checks that evaluation data and metrics are loaded
   - Prints diagnostic information:
     - Whether client is None
     - Client type (openai/databricks)
     - Model name
     - Number of samples (should be 10)
     - Number of metrics (should be 3)

2. **Helper Function: `safe_float`**:
   - Safely converts values to float type
   - Handles pandas NaN/None values
   - Returns default value (0.0) if conversion fails
   - Prevents crashes from invalid data types

3. **Instantiate Evaluator**:
   - Creates `LLMJudgeEvaluator` instance with:
     - Configured client
     - Selected judge model
     - List of MetricConfig objects (converted from JSON)
     - Ground truth DataFrame
     - Client type identifier

4. **Run Evaluation Loop**:
   - Iterates through each sample in `EVALUATION_DATA`
   - For each sample:
     - Converts DataFrame row to dictionary
     - Calls `evaluator.evaluate(sample)`
     - Receives back list of metric results
     - Appends results to `all_results` list

5. **Process Results**:
   - Converts `all_results` list into pandas DataFrame (`results_df`)
   - Calculates summary statistics:
     - Total evaluations performed
     - Number of passed evaluations
     - Number of failed evaluations
     - Pass rate percentage
   - Prints formatted results table

6. **Display Output**:
   - Shows each individual evaluation result:
     - Sample index
     - Metric name
     - LLM judge score
     - Pass/fail status
     - Ground truth value
     - Whether LLM matched ground truth
   - Summary statistics at the end

**Expected Output:**

For 10 samples × 3 metrics = 30 total evaluations, you should see:

```
[LLM CALL START] - For each evaluation
Raw response from LLM
[LLM CALL END]
Parsed score

... repeated 30 times ...

Results Summary:
- Total Evaluations: 30
- Passed: ~20-25
- Failed: ~5-10
- Pass Rate: ~70-80%
```

**Why This Matters:**
- This is where the actual evaluation happens
- Results are stored in `results_df` for use in next cell
- Debug output helps identify issues with LLM calls
- Summary statistics provide quick evaluation overview
- Pass/fail tracking enables quality assessment

**Troubleshooting:**
- If no "[LLM CALL START]" appears: LLM judge not being called
- If all scores are 0.0: Response parsing failing
- If error messages appear: Check API credentials
- If very slow: LLM API calls are sequential (normal)

**Time Expectation:**
- OpenAI GPT-4: ~2-4 seconds per call → ~2 minutes total
- GPT-3.5-turbo: ~0.5-1 second per call → ~30-60 seconds total
- Databricks models: Varies based on endpoint configuration

In [0]:
import time

print("="*80)
print("CELL 6: RUN EVALUATION")
print("="*80)

# Check prerequisites
print("\nChecking prerequisites...")
print(f"  client is None: {client is None}")
if client is not None:
    print(f"  client_type: {client_type}")
    print(f"  model: {JUDGE_MODEL}")
print(f"  Samples: {len(EVALUATION_DATA)}")
print(f"  Metrics: {len(METRICS_CONFIG_DATA)}")

# Helper function
def safe_float(value, default=0.0):
    """Safely convert to float."""
    if pd.isna(value) or value is None:
        return default
    try:
        val_str = str(value).strip().lower()
        if val_str in ['true', '==true', 'yes', '1']:
            return 1.0
        if val_str in ['false', '==false', 'no', '0']:
            return 0.0
        if val_str.endswith('%'):
            return float(val_str[:-1])
        return float(val_str)
    except:
        return default

def generate_prompt(name, description, rubric):
    """Generate evaluation prompt."""
    return f"""You are an expert evaluator. Task: {description}

Grading Rubric:
{rubric}

Evaluation Details:
- User Query: {{prompt}}
- AI Response: {{response}}
- Ground Truth: {{ground_truth}}

IMPORTANT: Respond with ONLY valid JSON (no other text):
{{
  "score": <your_numeric_score>,
  "explanation": "Brief explanation"
}}"""

# Load metrics
print("\nLoading metrics...")
metric_configs = []

for idx, row in METRICS_CONFIG_DATA.iterrows():
    name = str(row['name']).strip()
    type_str = str(row['type']).strip().lower()
    
    if type_str in ['binary', 'bool']:
        mtype = MetricType.BINARY
    elif type_str in ['1-5_scale', 'scale']:
        mtype = MetricType.SCALE_1_5
    else:
        mtype = MetricType.PERCENTAGE
    
    description = str(row.get('description', '')).strip()
    rubric = str(row.get('grading_rubric', '')).strip()
    threshold = safe_float(row.get('threshold', 0.5))
    
    prompt_template = generate_prompt(name, description, rubric)
    
    metric_config = MetricConfig(
        name=name,
        metric_type=mtype,
        description=description,
        prompt_template=prompt_template,
        threshold=threshold,
        ground_truth_column=str(row.get('ground_truth_column', '')).strip(),
        ground_truth_file_path=str(row.get('ground_truth_file_path', '')).strip()
    )
    
    metric_configs.append(metric_config)
    print(f"  Loaded: {name} ({mtype.value}, threshold={threshold})")

print(f"\nTotal metrics loaded: {len(metric_configs)}")

# Initialize evaluator
if client is None:
    print("\nERROR: Client not initialized!")
else:
    print("\nInitializing evaluator...")
    evaluator = LLMJudgeEvaluator(
        client=client,
        model=JUDGE_MODEL,
        metrics=metric_configs,
        ground_truth_data=GROUND_TRUTH_DATA,
        client_type=client_type
    )
    
    print(f"Evaluator ready")
    print(f"\n{'='*80}")
    print("ABOUT TO START EVALUATION")
    print(f"{'='*80}")
    print("Watch for [LLM CALL START] messages below...")
    print("If you don't see them, the LLM is not being called!")
    print(f"{'='*80}\n")
    
    # Force flush
    import sys
    sys.stdout.flush()
    
    # Run evaluation
    start_time = time.time()
    results_df = evaluator.evaluate_dataset(EVALUATION_DATA)
    eval_time = time.time() - start_time
    
    # Results
    print(f"\n{'='*80}")
    print(f"EVALUATION COMPLETE in {eval_time:.1f}s")
    print(f"{'='*80}")
    
    if eval_time < 5.0:
        print(f"\nWARNING: Evaluation was very fast ({eval_time:.1f}s)")
        print(f"Expected time for {len(results_df)} LLM calls: 20-60 seconds")
        print(f"This suggests LLM calls are failing silently!")
    
    # Analyze results
    total = len(results_df)
    passed = len(results_df[results_df['status'] == 'PASS'])
    failed = len(results_df[results_df['status'] == 'FAIL'])
    pass_rate = (passed / total * 100) if total > 0 else 0
    
    print(f"\nRESULTS SUMMARY:")
    print(f"  Total: {total}")
    print(f"  Passed: {passed}")
    print(f"  Failed: {failed}")
    print(f"  Pass rate: {pass_rate:.1f}%")
    print(f"  Time: {eval_time:.1f}s")
    
    # Check for all zeros
    if results_df['score'].sum() == 0:
        print(f"\nWARNING: All scores are 0!")
        print(f"This means:")
        print(f"  1. LLM calls are failing, OR")
        print(f"  2. LLM responses cannot be parsed, OR")
        print(f"  3. There's an error in evaluate_single")
        print(f"\nCheck the debug output above for [LLM CALL START] messages")
    
    # Show first few results
    print(f"\nFirst 3 results:")
    for i in range(min(3, len(results_df))):
        row = results_df.iloc[i]
        print(f"\n  {i+1}. Sample {row['sample_id']} - {row['metric_name']}:")
        print(f"     Score: {row['score']:.2f}")
        print(f"     Status: {row['status']}")
        print(f"     Explanation: {row['explanation'][:150]}")
    
    # Per-metric
    print(f"\nPer-metric results:")
    for metric_name in results_df['metric_name'].unique():
        metric_results = results_df[results_df['metric_name'] == metric_name]
        m_passed = len(metric_results[metric_results['status'] == 'PASS'])
        m_total = len(metric_results)
        m_rate = (m_passed / m_total * 100) if m_total > 0 else 0
        m_avg = metric_results['score'].mean()
        
        print(f"  {metric_name}: {m_rate:.1f}% pass ({m_passed}/{m_total}), avg score: {m_avg:.2f}")
    
    # Display full results
    print(f"\nDetailed results table:")
    display(results_df[['sample_id', 'metric_name', 'score', 'threshold', 'status', 'explanation']])
    
    print(f"\n{'='*80}")
    print("END OF CELL 6")
    print(f"{'='*80}")

CELL 6: RUN EVALUATION

Checking prerequisites...
  client is None: False
  client_type: databricks
  model: databricks-llm
  Samples: 10
  Metrics: 3

Loading metrics...
  Loaded: Clarity (1-5_scale, threshold=4.0)
  Loaded: Feasibility (binary, threshold=1.0)
  Loaded: Dietary_Fit (percentage, threshold=75.0)

Total metrics loaded: 3

Initializing evaluator...
[INIT] Evaluator created: databricks, 3 metrics
Evaluator ready

ABOUT TO START EVALUATION
Watch for [LLM CALL START] messages below...
If you don't see them, the LLM is not being called!


STARTING EVALUATION
Samples: 10
Metrics: 3
Total evaluations: 30

Sample 1/10: ID 1
  [  3.3%] Clarity...

    [LLM CALL START]
      Client type: databricks
      Model: databricks-llm
      Prompt length: 939 chars
      Making API call...
      Calling Databricks endpoint: databricks-claude-sonnet-4-5
      API call SUCCESS!
      Response length: 287 chars
      Response preview: ```json
{
  "score": 5,
  "explanation": "The instructions

sample_id,metric_name,score,threshold,status,explanation
1,Clarity,5.0,4.0,PASS,"The instructions are crystal clear with numbered steps (1-4) in logical order: seasoning, searing, adding vegetables with liquid, and finishing. Each step is concise and follows the natural cooking sequence, making it very easy to follow."
1,Feasibility,1.0,1.0,PASS,"The recipe uses common ingredients (chicken thighs, broccoli, lemon, garlic, broth, butter) all available at standard grocery stores. Total cooking time is approximately 15-20 minutes (4-5 min searing per side + 4-6 min covered cooking), well within the 30-minute weeknight constraint. This passes the feasibility criteria."
1,Dietary_Fit,75.0,75.0,PASS,"The recipe is mostly kid-friendly with lean protein (chicken) and vegetables (broccoli). However, lemon-garlic flavors may be strong for some children. Easy swaps available: milder seasonings, cheese sauce for broccoli, or honey glaze. The addition of rice provides balanced carbohydrates. Minor concerns: bone-in thighs pose choking hazards (boneless recommended), and butter adds saturated fat. Overall appropriate with simple modifications for pickier eaters."
2,Clarity,4.0,4.0,PASS,"The instructions are mostly clear and logically ordered with numbered steps. The sequence flows well (boil pasta, make sauce, combine). However, some details could be clearer: cooking times aren't specified for boiling or sautéing, and 'add tomato sauce' could specify whether it's jarred or homemade. The steps are easy to follow overall but lack precision in a few areas."
2,Feasibility,1.0,1.0,PASS,"The recipe passes feasibility criteria: penne pasta, carrots, zucchini, tomato sauce, cream cheese, and parmesan are all common grocery store ingredients. The cooking process (boiling pasta, sautéing vegetables, mixing sauce) can easily be completed in approximately 30 minutes, making it suitable for a weeknight dinner."
2,Dietary_Fit,85.0,75.0,PASS,"The AI response is highly suitable for kids: it's vegetarian as requested, avoids mushrooms, includes hidden vegetables (carrot, zucchini) for nutrition, uses kid-friendly pasta and creamy sauce, and adds fresh cucumbers. The recipe is simple and balanced. Minor deduction because cream cheese adds richness but could be swapped for a lighter option, and the ground truth recipe (sausage-based) doesn't match the vegetarian request, making the AI response actually more appropriate for the query."
4,Clarity,4.0,4.0,PASS,"The instructions are mostly clear and logically ordered with numbered steps. The sequence is intuitive (scramble, wilt, assemble, roll). However, some minor details could be clearer, such as whether to mix eggs and spinach together or layer them separately, and the exact timing for each step. The optional salsa addition is appropriately noted. Overall, easy to follow for a quick recipe."
4,Feasibility,1.0,1.0,PASS,"The recipe passes feasibility criteria: 10-minute cook time is well under 30 minutes, and all ingredients (eggs, spinach, tortillas, yogurt, salsa) are common grocery store items. This clearly meets the weeknight cooking standard."
4,Dietary_Fit,100.0,75.0,PASS,"The egg and spinach tortilla wraps are perfectly kid-friendly with balanced nutrition: protein from eggs, vegetables (spinach), whole grains (tortillas), and optional dairy (yogurt). The recipe is simple, quick, and offers easy swaps (cheese instead of yogurt, different veggies). It provides essential nutrients children need and can be adapted for picky eaters."
6,Clarity,5.0,4.0,PASS,"The instructions are crystal clear with numbered steps in logical order: prep tofu, cook tofu, add vegetables and sauce, serve. Each step is concise and actionable, making it very easy to follow."



END OF CELL 6


### 📈 **Detailed Explanation: Cell 7 - MLflow Logging and Visualization**

This is the final cell that logs all evaluation results to MLflow and creates interactive visualizations for analysis. It provides comprehensive tracking and visual insights into model performance.

**What This Cell Does:**

1. **Import Required Libraries**:
   - MLflow for experiment tracking
   - Plotly for interactive visualizations
   - Pandas/NumPy for data manipulation

2. **Check for Results**:
   - Verifies `results_df` exists from previous cell
   - Confirms DataFrame is not empty
   - Validates it contains expected columns

3. **Data Transformation**:
   - **Pivot Table Creation**: Converts long-format results to wide format
     - Rows: Sample index
     - Columns: Metric names (Clarity, Safety, Relevance)
     - Values: LLM judge scores
   - **Adds Metadata Columns**:
     - Original request and response text
     - Overall pass/fail status per sample
     - Timestamp information

4. **MLflow Experiment Setup**:
   - Sets experiment name: `/Users/[your-email]/llm-judge-evaluation`
   - Creates new run for this evaluation session
   - Logs run metadata:
     - Judge model used
     - Number of samples evaluated
     - Number of metrics
     - Client type (OpenAI/Databricks)

5. **Metrics Logging**:
   - **Per-Metric Statistics**:
     - Mean score for each metric
     - Min/max scores
     - Pass rate percentage
     - Standard deviation
   - **Overall Statistics**:
     - Total pass rate across all metrics
     - Number of perfect scores
     - Number of complete failures
     - Accuracy vs ground truth (if available)

6. **Interactive Visualizations** (using Plotly):
   
   **Chart 1: Score Distribution by Metric**
   - Box plot showing score ranges
   - Violin plot overlays for distribution shape
   - Helps identify which metrics are hardest/easiest to pass
   
   **Chart 2: Pass Rate by Metric**
   - Bar chart with pass rates
   - Shows threshold line
   - Color-coded by performance
   
   **Chart 3: Score Heatmap**
   - Rows = samples, Columns = metrics
   - Color intensity = score value
   - Quickly spot problem areas
   
   **Chart 4: Ground Truth Comparison** (if available)
   - Scatter plot: LLM score vs. expected score
   - Diagonal line = perfect agreement
   - Correlation coefficient displayed
   - Helps assess judge accuracy

7. **Artifact Logging**:
   - Saves results DataFrame as CSV
   - Saves each visualization as interactive HTML
   - Logs all to MLflow for permanent storage
   - Creates downloadable reports

8. **Final Output**:
   - Displays all charts inline in notebook
   - Prints summary statistics
   - Provides MLflow run URL for detailed tracking
   - Shows where artifacts are stored

**Why This Matters:**
- **Experiment Tracking**: All runs logged for comparison
- **Reproducibility**: Complete record of evaluation conditions
- **Visual Analysis**: Quickly identify patterns and issues
- **Reporting**: Shareable HTML reports for stakeholders
- **Historical Tracking**: Compare model performance over time
- **Debugging**: Detailed logs help troubleshoot evaluation issues

**MLflow Benefits:**
- **Centralized Storage**: All results in one place
- **Version Control**: Track changes to evaluation setup
- **Comparison**: Compare different judge models or thresholds
- **Collaboration**: Share results with team
- **Audit Trail**: Complete history of evaluations

**Generated Artifacts:**
1. `results.csv` - Full evaluation results table
2. `score_distribution.html` - Interactive box/violin plot
3. `pass_rates.html` - Bar chart of pass rates
4. `score_heatmap.html` - Color-coded performance matrix
5. `ground_truth_comparison.html` - Accuracy assessment plot

**Use Cases:**
- Present evaluation results to stakeholders
- Compare different LLM judges (GPT-4 vs GPT-3.5)
- Adjust thresholds based on score distributions
- Identify which metrics need improvement
- Track model quality over time
- Generate compliance reports

In [0]:
# ===== Import results (if present) and then run MLflow logging + charts (DYNAMIC METRICS) =====
# - Uses an existing `results_df` if defined and non-empty.
# - If it looks like judge outputs (metric_name/score), pivots dynamically to wide format by metric.
# - Discovers ALL metrics from METRICS_CONFIG_DATA (no hardcoding).
# - Interactive Plotly visuals; artifacts logged to MLflow as HTML (no filesystem writes).

import os, re, sys, time
import pandas as pd
import numpy as np
import mlflow
import plotly.graph_objects as go
import plotly.io as pio

# -------------------- Helpers --------------------
def _load_results_df():
    if "results_df" in globals():
        try:
            if isinstance(results_df, pd.DataFrame) and not results_df.empty:
                print("Using existing `results_df` from memory.")
                return results_df.copy()
        except Exception:
            pass
    raise FileNotFoundError("No results found in memory. Ensure `results_df` is created in earlier cells.")

def _extract_title_from_text(text: str):
    if not isinstance(text, str) or not text.strip():
        return None
    m = re.search(r"(?i)title:\s*(.+?)(?:\.|\n|$)", text)
    if m:
        return m.group(1).strip()[:120]
    return text.strip().split("\n")[0][:120]

def _sanitize_key(s: str) -> str:
    # Good for MLflow metric keys
    return re.sub(r"[^A-Za-z0-9_]+", "_", str(s))

def _coerce_binary(series: pd.Series) -> pd.Series:
    # Accept 1/0, True/False, Pass/Fail, Yes/No
    mapping = {
        "pass": 1, "fail": 0,
        "true": 1, "false": 0,
        "yes": 1, "no": 0
    }
    s = series.astype(str).str.strip().str.lower().map(lambda v: mapping.get(v, v))
    return pd.to_numeric(s, errors="coerce")

def _coerce_percentage(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce")
    # if values look like 0..1, treat as proportions
    if s.dropna().between(0, 1).mean() > 0.5:
        s = s * 100.0
    return s

def _find_metric_column(df: pd.DataFrame, metric_name: str, gt_col_hint: str | None) -> str | None:
    """
    Try several reasonable column names for a metric:
    - exact ground_truth_column
    - exact metric name
    - snake and lowercase variants
    - common suffixes: _score, _pct, _percent, _pass
    """
    candidates: list[str] = []
    if gt_col_hint:
        candidates.append(gt_col_hint)

    # Build deterministic name variants (preserve priority + de-dupe)
    base_names = [
        metric_name,
        metric_name.replace(" ", "_"),
        metric_name.replace("-", "_"),
        metric_name.lower(),
        metric_name.lower().replace(" ", "_").replace("-", "_"),
    ]
    name_variants = []
    for n in base_names:
        if n not in name_variants:
            name_variants.append(n)

    suffixes = ["", "_score", "_pct", "_percent", "_pass"]
    for base in name_variants:
        for suf in suffixes:
            candidates.append(f"{base}{suf}")

    # Pick first present (case-sensitive first)
    for c in candidates:
        if c in df.columns:
            return c
    # Case-insensitive fallback
    lower_map = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]
    return None

# -------------------- Load & standardize --------------------
assert "METRICS_CONFIG_DATA" in globals() and isinstance(METRICS_CONFIG_DATA, pd.DataFrame), \
    "Run your setup cell first to populate METRICS_CONFIG_DATA."

df_results = _load_results_df()
print(f"✅ Loaded results shape: {df_results.shape}")

# If this looks like judge outputs (metric_name/score), pivot to wide
needs_pivot = {"metric_name", "score", "sample_id"}.issubset(df_results.columns)
if needs_pivot:
    print("ℹ️ Detected judge-output format — pivoting wide by metric_name.")
    dfj = df_results.copy()
    dfj["score"] = pd.to_numeric(dfj["score"], errors="coerce")

    pvt = dfj.pivot_table(index="sample_id", columns="metric_name", values="score", aggfunc="mean")
    pvt = pvt.rename_axis(None, axis=1).reset_index()

    # recipe_title if we can derive from EVALUATION_DATA
    if "EVALUATION_DATA" in globals() and isinstance(EVALUATION_DATA, pd.DataFrame):
        title_df = EVALUATION_DATA[["sample_id", "prompt", "response"]].copy()
        title_df["recipe_title"] = title_df["response"].apply(_extract_title_from_text)
        title_df["recipe_title"] = title_df["recipe_title"].fillna(title_df["prompt"].apply(_extract_title_from_text))
        title_df["recipe_title"] = title_df.apply(
            lambda r: r["recipe_title"] if isinstance(r["recipe_title"], str) and r["recipe_title"].strip()
            else f"Sample {r['sample_id']}", axis=1)
        pvt = pvt.merge(title_df[["sample_id", "recipe_title"]], on="sample_id", how="left")
    if "recipe_title" not in pvt.columns or pvt["recipe_title"].isna().any():
        pvt["recipe_title"] = pvt["sample_id"].apply(lambda x: f"Sample {int(x)}")

    df_results = pvt.copy()
    print(f"✅ Pivot complete. Columns now: {list(df_results.columns)[:8]} ...")

# Make sure we have sample_id and a title
if "sample_id" not in df_results.columns:
    df_results["sample_id"] = np.arange(1, len(df_results) + 1)
if "recipe_title" not in df_results.columns:
    # Try to synthesize from response/prompt if present
    if {"response", "prompt"}.issubset(df_results.columns):
        df_results["recipe_title"] = df_results.apply(
            lambda r: _extract_title_from_text((r.get("response") or r.get("prompt") or "")) or f"Sample {r['sample_id']}",
            axis=1
        )
    else:
        df_results["recipe_title"] = df_results["sample_id"].apply(lambda x: f"Sample {int(x)}")

# -------------------- Build dynamic metric values & pass flags --------------------
metrics = []
for _, row in METRICS_CONFIG_DATA.iterrows():
    name = str(row.get("name", "")).strip()
    mtype = str(row.get("type", "")).strip().lower()
    thr_raw = row.get("threshold", "")
    gt_col = str(row.get("ground_truth_column", "")).strip() or None
    if not name:
        continue

    # threshold numeric
    try:
        thr = float(str(thr_raw).replace("%", "").strip())
    except:
        thr = 0.0

    # locate column for this metric
    col = _find_metric_column(df_results, name, gt_col)
    if col is None:
        print(f"⚠️ Skipping metric '{name}': no matching column found in results.")
        continue

    s = df_results[col]

    # normalize by type
    if mtype in ["binary", "bool", "boolean"]:
        vals = _coerce_binary(s)
        # if still NaN, treat as 0
        vals = vals.fillna(0)
        pass_flag = (vals >= thr).astype(int)  # typically thr=1
        score_for_stats = vals.copy()

    elif mtype in ["1-5_scale", "scale", "1_5_scale"]:
        vals = pd.to_numeric(s, errors="coerce")
        pass_flag = (vals >= thr).astype(int)
        score_for_stats = vals.copy()

    else:  # percentage
        vals = _coerce_percentage(s)
        pass_flag = (vals >= thr).astype(int)
        score_for_stats = vals.copy()

    metrics.append({
        "name": name,
        "type": mtype,
        "threshold": thr,
        "column": col,
        "values": vals,
        "pass_flag": pass_flag,
        "score_for_stats": score_for_stats
    })

if not metrics:
    raise ValueError("No usable metrics found. Ensure METRICS_CONFIG_DATA lines up with columns or judge-output names.")

# assemble standardized frame for charts
std = df_results[["sample_id", "recipe_title"]].copy()
for m in metrics:
    safe = _sanitize_key(m["name"])
    std[f"{safe}__value"] = m["values"]
    std[f"{safe}__pass"]  = m["pass_flag"]

# overall pass = all metrics passed (row-wise)
pass_cols = [f"{_sanitize_key(m['name'])}__pass" for m in metrics]
std["pass_all"] = std[pass_cols].all(axis=1).astype(int)

# -------------------- Aggregate stats & MLflow logging --------------------
overall_pass_rate = float(std["pass_all"].mean()) if len(std) else 0.0

metric_pass_rates = {}
metric_means = {}
failure_counts = {}
for m in metrics:
    safe = _sanitize_key(m["name"])
    metric_pass_rates[m["name"]] = float(std[f"{safe}__pass"].mean())
    # mean of the raw/normalized values
    metric_means[m["name"]] = float(pd.to_numeric(std[f"{safe}__value"], errors="coerce").mean())
    failure_counts[m["name"]] = int((std[f"{safe}__pass"] == 0).sum())

with mlflow.start_run(run_name="Judge Analysis (Dynamic)"):
    # Log thresholds as params & dynamic metrics
    mlflow.log_metric("overall_pass_rate", overall_pass_rate)
    for m in metrics:
        safe = _sanitize_key(m["name"])
        mlflow.log_param(f"{safe}_threshold", m["threshold"])
        mlflow.log_metric(f"{safe}_pass_rate", metric_pass_rates[m["name"]])
        mlflow.log_metric(f"{safe}_mean", metric_means[m["name"]])

    # Also log the standardized table (values + pass flags) as CSV text
    mlflow.log_text(std.to_csv(index=False), "standardized_results_dynamic.csv")

    # -------------------- Plotly: Overall Pass Gauge --------------------
    gauge_val = overall_pass_rate * 100.0
    fig_gauge = go.Figure(go.Indicator(
        mode="gauge+number+delta",
        value=gauge_val,
        number={'suffix': "%", 'font': {'size': 36}},
        delta={'reference': 70, 'increasing': {'color': '#2ecc71'}, 'decreasing': {'color': '#e74c3c'}},
        gauge={
            'axis': {'range': [0, 100]},
            'bar': {'color': '#34495e', 'thickness': 0.25},
            'steps': [
                {'range': [0, 50], 'color': '#fde2e1'},
                {'range': [50, 70], 'color': '#fff2cc'},
                {'range': [70, 100], 'color': '#e3f5e5'}
            ],
            'threshold': {'line': {'color': '#1abc9c', 'width': 4}, 'thickness': 0.75, 'value': 70}
        },
        title={'text': "Overall Pass Rate", 'font': {'size': 20}}
    ))
    fig_gauge.update_layout(height=320, margin=dict(l=40, r=40, t=60, b=20))
    displayHTML(pio.to_html(fig_gauge, include_plotlyjs='cdn'))
    mlflow.log_text(pio.to_html(fig_gauge, include_plotlyjs='cdn'), "overall_pass_rate.html")

    # -------------------- Plotly: Per-Recipe Pass/Fail Heatmap (dynamic columns) --------------------
    metric_names = [m["name"] for m in metrics]
    z = np.column_stack([std[f"{_sanitize_key(n)}__pass"].to_numpy() for n in metric_names])  # shape (n_rows, n_metrics)
    ztext = np.where(z == 1, "Pass", "Fail")

    fig_heatmap = go.Figure(data=go.Heatmap(
        z=z.T,  # metrics on y (rows), samples on x
        x=[str(s) for s in std["sample_id"]],
        y=metric_names,
        colorscale=[[0, '#e76f51'], [1, '#2a9d8f']],
        zmin=0, zmax=1,
        text=ztext.T,
        texttemplate="%{text}",
        colorbar=dict(title="Status", tickvals=[0,1], ticktext=["Fail","Pass"])
    ))
    fig_heatmap.update_layout(
        title={'text': "Per-Recipe Metric Pass/Fail (interactive)", 'x': 0.5},
        xaxis_title="Sample ID",
        yaxis_title="Metric",
        height=max(420, 30 * len(metric_names)),
        width=max(900, 28 * len(std)),
        margin=dict(l=80, r=40, t=60, b=60)
    )
    displayHTML(pio.to_html(fig_heatmap, include_plotlyjs='cdn'))
    mlflow.log_text(pio.to_html(fig_heatmap, include_plotlyjs='cdn'), "per_recipe_metric_heatmap.html")

    # -------------------- Plotly: Metric Failure Frequency (sorted bar) --------------------
    fail_items = sorted(failure_counts.items(), key=lambda x: x[1], reverse=True)
    fig_fail = go.Figure(go.Bar(
        x=[k for k, _ in fail_items],
        y=[v for _, v in fail_items],
        marker_color=['#e74c3c' if v > 0 else '#2ecc71' for _, v in fail_items],
        text=[str(v) for _, v in fail_items],
        textposition="outside",
        hovertemplate="Metric: %{x}<br>Failures: %{y}<extra></extra>"
    ))
    fig_fail.update_layout(
        title={'text': "Metric Failures (count)", 'x': 0.5},
        yaxis_title="Failures",
        xaxis_title="Metric",
        height=400,
        margin=dict(l=60, r=40, t=60, b=60)
    )
    displayHTML(pio.to_html(fig_fail, include_plotlyjs='cdn'))
    mlflow.log_text(pio.to_html(fig_fail, include_plotlyjs='cdn'), "metric_fail_counts.html")

    # -------------------- Plotly: Pass Rates by Metric (dynamic) --------------------
    pass_rate_pct = [metric_pass_rates[n]*100 for n in metric_names]
    fig_rates = go.Figure(go.Bar(
        x=metric_names,
        y=pass_rate_pct,
        text=[f"{v:.1f}%" for v in pass_rate_pct],
        textposition="outside",
        marker_color=['#3498db']*len(metric_names),
        hovertemplate="Metric: %{x}<br>Pass Rate: %{y:.1f}%<extra></extra>"
    ))
    fig_rates.update_layout(
        title={'text': "Pass Rates by Metric", 'x': 0.5},
        yaxis=dict(title="Pass Rate (%)", range=[0,100]),
        height=max(420, 40*len(metric_names)),
        margin=dict(l=60, r=40, t=60, b=120),
        xaxis={'tickangle': -30}
    )
    displayHTML(pio.to_html(fig_rates, include_plotlyjs='cdn'))
    mlflow.log_text(pio.to_html(fig_rates, include_plotlyjs='cdn'), "metric_pass_rates.html")

print("\n=== SUMMARY ===")
print(f"Overall pass rate: {overall_pass_rate:.1%}")
print("Pass rates by metric:")
for k, v in metric_pass_rates.items():
    print(f"  - {k}: {v:.1%}")
print("Fail counts by metric:")
for k, v in failure_counts.items():
    print(f"  - {k}: {v}")


Using existing `results_df` from memory.
✅ Loaded results shape: (30, 8)
ℹ️ Detected judge-output format — pivoting wide by metric_name.
✅ Pivot complete. Columns now: ['sample_id', 'Clarity', 'Dietary_Fit', 'Feasibility', 'recipe_title'] ...



=== SUMMARY ===
Overall pass rate: 60.0%
Pass rates by metric:
  - Clarity: 80.0%
  - Feasibility: 70.0%
  - Dietary_Fit: 80.0%
Fail counts by metric:
  - Clarity: 2
  - Feasibility: 3
  - Dietary_Fit: 2


## Debug Guide

**What to look for in Cell 6 output:**

1. **[LLM CALL START]** - Should appear for each evaluation (9 times for 3 samples x 3 metrics)
   - If missing: LLM is NOT being called

2. **API call SUCCESS!** - Should appear after each LLM call
   - If missing: API calls are failing

3. **Evaluation time** - Should be 20-60 seconds for 9 LLM calls
   - If < 5 seconds: LLM calls are failing silently

4. **All scores are 0** warning - Indicates parsing or API issues

5. **ERROR messages** - Will show full traceback of any failures